In [ ]:
## Register Model
from azureml.core import Workspace, Model

ws = Workspace.from_config()

model = Model.register(
    workspace=ws,
    model_path="model.pkl",  # local path
    model_name="housing-price-model"  # name in Azure ML
)


In [ ]:
## Create inference config
from azureml.core.environment import Environment
from azureml.core.model import InferenceConfig

env = Environment.from_conda_specification(name="housing-env", file_path="environment.yml")

inference_config = InferenceConfig(
    entry_script="score.py",
    environment=env
)


In [ ]:
## Deploy as WEb SErvice
from azureml.core.webservice import AciWebservice, Webservice

deployment_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)

service = Model.deploy(
    workspace=ws,
    name="housing-price-service",
    models=[model],
    inference_config=inference_config,
    deployment_config=deployment_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)
print(service.state)


In [ ]:
## TEst endpoint
import requests
import json

scoring_uri = service.scoring_uri

# Example input
input_data = {
    "data": [[3000, 3, 2, 2, 1]]  # area, bedrooms, bathrooms, stories, parking
}

headers = {"Content-Type": "application/json"}
response = requests.post(scoring_uri, data=json.dumps(input_data), headers=headers)

print("Prediction:", response.json())
